# 8th attempt – Count Regression

**Goal:** Given the *current* 10×10 Game-of-Life board, predict the **total number of live cells** in the *previous* board.

This is a **regression** problem (output: a single integer 0-100), unlike the earlier per-cell classification notebooks.

| | Earlier notebooks | This notebook |
|---|---|---|
| Task | Per-cell binary classification | Whole-board live-cell count regression |
| Output | 0 / 1 (dead / alive) | scalar (0 – 100) |
| Loss | binary cross-entropy | MAE |
| Metric | accuracy | MAE in cells, % within N cells |

In [ ]:
import functions
from functions import *
import importlib
importlib.reload(functions)
from functions import *

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from functions import (
    load_reverse_df,
    prepare_count_regression_dataset,
    to_numpy_count_regression,
    build_and_train_count_regression_cnn,
    evaluate_count_regression,
)

In [ ]:
SIZE          = 10
AMOUNT_BOARDS = 10000
AMOUNT_MOVES  = 100
NUM_DICT      = 1
gen           = 2   # 1 current board + 1 previous board stored per row

## 1. Load data

In [ ]:
reverse_df = load_reverse_df(SIZE, AMOUNT_BOARDS, gen)
print("DataFrame shape:", reverse_df.shape)

## 2. Prepare dataset

- **X** = first 100 columns → the *current* board  
- **y** = sum of columns 100-199 → live-cell count in the *previous* board

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = prepare_count_regression_dataset(
    reverse_df, SIZE, gen, test_size=0.1, val_size=0.1, random_state=365
)

print(f"Train : {len(X_train):>7,}  samples")
print(f"Val   : {len(X_val):>7,}  samples")
print(f"Test  : {len(X_test):>7,}  samples")
print(f"\nTarget range  min={y_train.min():.0f}  max={y_train.max():.0f}  mean={y_train.mean():.1f}")

In [ ]:
# Quick look at the label distribution
plt.figure(figsize=(8, 3))
y_train.hist(bins=50, edgecolor='black')
plt.title('Distribution of previous-board live-cell count (train set)')
plt.xlabel('Live cells in previous board')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

## 3. Convert to numpy (CNN-ready)

In [ ]:
X_train_arr, X_val_arr, X_test_arr, \
y_train_arr, y_val_arr, y_test_arr = to_numpy_count_regression(
    X_train, X_val, X_test,
    y_train, y_val, y_test,
    SIZE
)

print("X_train shape:", X_train_arr.shape)   # (N, 10, 10, 1)
print("y_train shape:", y_train_arr.shape)   # (N, 1)

## 4. Build and train the model

Architecture: `Conv2D(32) → Conv2D(64) → Conv2D(128)` + BatchNorm after each  
→ `Flatten → Dense(256) → Dropout(0.3) → Dense(128) → Dropout(0.2) → Dense(1)`

Loss: **MAE** (interpretable directly as cells off on average).

In [ ]:
model, history = build_and_train_count_regression_cnn(
    X_train_arr,
    y_train_arr,
    size=SIZE,
    epochs=30,
    batch_size=256,
    validation_split=0.2,
)

model.summary()

## 5. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['mae'],     label='train MAE')
axes[0].plot(history.history['val_mae'], label='val MAE')
axes[0].set_title('MAE over epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MAE (cells)')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='train loss')
axes[1].plot(history.history['val_loss'], label='val loss')
axes[1].set_title('Loss over epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Evaluate on test set

In [ ]:
test_loss = model.evaluate(X_test_arr, y_test_arr, verbose=0)
print(f"Test MAE : {test_loss[1]:.3f} cells")
print(f"Test MSE : {test_loss[2]:.3f}")

In [ ]:
y_pred, y_true = evaluate_count_regression(model, X_test_arr, y_test_arr)

## 7. Predicted vs actual scatter

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_true, y_pred, alpha=0.2, s=5, label='samples')
lim = [0, SIZE * SIZE]
plt.plot(lim, lim, 'r--', linewidth=1.5, label='perfect prediction')
plt.xlabel('True live-cell count (previous board)')
plt.ylabel('Predicted live-cell count')
plt.title('Predicted vs True – count regression')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Error distribution

In [ ]:
errors = y_pred - y_true

plt.figure(figsize=(8, 3))
plt.hist(errors, bins=60, edgecolor='black')
plt.axvline(0, color='red', linestyle='--', linewidth=1.5, label='zero error')
plt.xlabel('Prediction error  (predicted − true)')
plt.ylabel('Frequency')
plt.title('Error distribution on test set')
plt.legend()
plt.tight_layout()
plt.show()